# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vagisha14/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rule: prioritize items that are visible but underperforming. The score goes up for low CTR, stale content, slipping position, weak conversions, and high bounce. Reason codes: low_ctr, stale_but_visible, position_slipping, weak_conversions, high_bounce_rate. This is decision-support only, not a final human decision.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ---------------------------
# 1) Find a starting dataframe
# ---------------------------
def find_input_df():
    # Use an already-loaded dataframe if the notebook has one
    for name in ["df", "data", "dataset", "items", "queue", "pages", "articles"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            return obj.copy(), f"globals()['{name}']"

    # Otherwise, look for the first usable CSV in the repo
    root = Path.cwd()
    csvs = [p for p in root.rglob("*.csv") if "baseline_action_score" not in p.name.lower()]
    if not csvs:
        raise FileNotFoundError("No input CSV found and no dataframe already loaded in memory.")

    # Prefer files inside work/ or data/
    csvs = sorted(csvs, key=lambda p: (0 if "work" in p.parts else 1 if "data" in p.parts else 2, len(p.parts), p.name))
    return pd.read_csv(csvs[0]), str(csvs[0])

df, source = find_input_df()
print("Loaded from:", source)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# ---------------------------
# 2) Helper functions
# ---------------------------
def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None

def numeric_series(df, col):
    s = pd.to_numeric(df[col], errors="coerce")
    if s.notna().sum() == 0:
        return None
    return s

def norm_0_1(s):
    s = s.astype(float)
    if s.notna().sum() == 0:
        return pd.Series(0.0, index=s.index)
    lo = s.min(skipna=True)
    hi = s.max(skipna=True)
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

# ---------------------------
# 3) Score using available signals
# ---------------------------
components = []
reason_map = []

# Low CTR is bad
ctr_col = pick_col(df, ["ctr", "click_through_rate", "clickthroughrate", "ctr_7d", "ctr_28d"])
if ctr_col:
    s = numeric_series(df, ctr_col)
    if s is not None:
        c = (1 - norm_0_1(s)).fillna(0.0)
        components.append(("low_ctr", c, 3.0))
        reason_map.append(("low_ctr", "Low CTR / weak click appeal"))

# Stale content is bad
stale_col = pick_col(df, ["days_since_update", "age_days", "stale_days", "days_old", "content_age_days"])
if stale_col:
    s = numeric_series(df, stale_col)
    if s is not None:
        c = norm_0_1(s).fillna(0.0)
        components.append(("stale_but_visible", c, 2.5))
        reason_map.append(("stale_but_visible", "Older item that is still visible"))

# High position/rank number is bad
pos_col = pick_col(df, ["position", "rank", "avg_position", "search_position"])
if pos_col:
    s = numeric_series(df, pos_col)
    if s is not None:
        c = norm_0_1(s).fillna(0.0)
        components.append(("position_slipping", c, 2.5))
        reason_map.append(("position_slipping", "Worse ranking / slipping position"))

# High bounce is bad
bounce_col = pick_col(df, ["bounce_rate", "exit_rate", "bounce"])
if bounce_col:
    s = numeric_series(df, bounce_col)
    if s is not None:
        c = norm_0_1(s).fillna(0.0)
        components.append(("high_bounce_rate", c, 2.0))
        reason_map.append(("high_bounce_rate", "High bounce / weak engagement"))

# Low conversions is bad
conv_col = pick_col(df, ["conversion_rate", "conv_rate", "orders_rate", "purchase_rate"])
if conv_col:
    s = numeric_series(df, conv_col)
    if s is not None:
        c = (1 - norm_0_1(s)).fillna(0.0)
        components.append(("weak_conversions", c, 2.0))
        reason_map.append(("weak_conversions", "Low conversion / weak action rate"))

# Low clicks on visible items can matter
impr_col = pick_col(df, ["impressions", "views", "traffic"])
click_col = pick_col(df, ["clicks", "click_count", "sessions_clicks"])
if impr_col and click_col:
    impr = numeric_series(df, impr_col)
    clicks = numeric_series(df, click_col)
    if impr is not None and clicks is not None:
        with np.errstate(divide="ignore", invalid="ignore"):
            c = (norm_0_1(impr) * (1 - norm_0_1(clicks))).fillna(0.0)
        components.append(("stale_but_visible", c, 1.5))
        reason_map.append(("stale_but_visible", "Visible item with limited response"))

if not components:
    raise ValueError(
        "No usable scoring columns found. Inspect df.columns and add the right column names to the alias lists."
    )

# Combine components into one score
raw = pd.Series(0.0, index=df.index)
for _, comp, weight in components:
    raw = raw + (comp * weight)

score = 100 * norm_0_1(raw)

# Primary reason = component with highest contribution for that row
component_frame = pd.DataFrame(index=df.index)
for name, comp, weight in components:
    component_frame[name] = comp * weight

primary_reason = component_frame.idxmax(axis=1)

# Action mapping
action_map = {
    "low_ctr": "Rewrite title/meta and sharpen the hook",
    "stale_but_visible": "Refresh the content and update details",
    "position_slipping": "Improve internal links and page relevance",
    "high_bounce_rate": "Tighten the opening and improve clarity",
    "weak_conversions": "Strengthen CTA and intent match",
}

confidence_note_map = {
    "low_ctr": "Strong when impressions are decent and CTR is clearly below peers.",
    "stale_but_visible": "Strong when the item is visible but older than the rest.",
    "position_slipping": "Strong when rank/position is clearly worse than peers.",
    "high_bounce_rate": "Strong when engagement is poor across the item.",
    "weak_conversions": "Strong when traffic exists but conversions lag.",
}

ranked_queue = df.copy()
ranked_queue["score"] = score.round(2)
ranked_queue["reason_code"] = primary_reason
ranked_queue["recommended_action"] = ranked_queue["reason_code"].map(action_map).fillna("Keep / monitor")
ranked_queue["confidence_note"] = ranked_queue["reason_code"].map(confidence_note_map).fillna("Use as a review signal only.")

# Sort highest score first
ranked_queue = ranked_queue.sort_values("score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = np.arange(1, len(ranked_queue) + 1)

# Save CSV
out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked_queue.to_csv(out_path, index=False)

print("Saved:", out_path)
print(ranked_queue[["rank", "score", "reason_code", "recommended_action"]].head(10))

Loaded from: /content/Internship/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Saved: work/outputs/baseline_action_score.csv
   rank   score reason_code                       recommended_action
0     1  100.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For the top 20, I will only recommend actions that match the strongest visible signal. Each recommendation is based on the score, the reason code, and the supporting column pattern. The note explains why the pick looks reasonable, and the wrong-case note explains what would make the pick misleading.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
# Top-20 review
top20 = ranked_queue.head(20).copy()

def wrong_case_text(reason):
    mapping = {
        "low_ctr": "This would be wrong if the item has strong CTR on a smaller but high-intent audience.",
        "stale_but_visible": "This would be wrong if the content was just refreshed outside the dataset window.",
        "position_slipping": "This would be wrong if the position column is stale or measured from a different snapshot.",
        "high_bounce_rate": "This would be wrong if bounce is caused by tracking noise or a landing-page mismatch outside the dataset scope.",
        "weak_conversions": "This would be wrong if conversion is delayed and not fully captured in the current window.",
    }
    return mapping.get(reason, "This would be wrong if the signal is noisy, missing, or not comparable across rows.")

top20_review = pd.DataFrame({
    "rank": top20["rank"],
    "score": top20["score"],
    "reason_code": top20["reason_code"],
    "action": top20["recommended_action"],
    "confidence_note": top20["confidence_note"],
    "what_would_make_it_wrong": top20["reason_code"].map(wrong_case_text),
})

print(top20_review.to_string(index=False))

# Leakage check
leak_markers = ["future", "next", "lookahead", "label", "target", "product_flag", "is_product", "window", "post_", "after_"]
leak_cols = [c for c in df.columns if any(m in c.lower() for m in leak_markers)]

print("\nLeakage check:")
if leak_cols:
    print("Potential leakage columns found:", leak_cols)
else:
    print("No obvious future-window or product-flag leakage columns found from column names.")

# Optional sanity check on the output file
assert Path("work/outputs/baseline_action_score.csv").exists(), "Output CSV was not created."

 rank  score reason_code                                  action                                                    confidence_note                                                              what_would_make_it_wrong
    1 100.00     low_ctr Rewrite title/meta and sharpen the hook Strong when impressions are decent and CTR is clearly below peers. This would be wrong if the item has strong CTR on a smaller but high-intent audience.
    2  93.53     low_ctr Rewrite title/meta and sharpen the hook Strong when impressions are decent and CTR is clearly below peers. This would be wrong if the item has strong CTR on a smaller but high-intent audience.
    3  93.23     low_ctr Rewrite title/meta and sharpen the hook Strong when impressions are decent and CTR is clearly below peers. This would be wrong if the item has strong CTR on a smaller but high-intent audience.
    4  92.08     low_ctr Rewrite title/meta and sharpen the hook Strong when impressions are decent and CTR is clearly below pee

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.